# parents-dict-by-argidx — worked example 1: Three-arg forward with scalars skipped in parents

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parents-dict-by-argidx`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `parents` dict for a computation node maps each positional argument index to the corresponding Tensor input, skipping all non-Tensor arguments (scalars, shapes, etc.). The key insight is that the keys are the ORIGINAL argument indices, not a re-numbered sequence. If arg 0 and arg 2 are Tensors but arg 1 is a float, then `parents = {0: t0, 2: t2}` — index 1 is entirely absent.

## Worked solution

**Step 1 — Identify the scenario.** We have a function called with `(tensor_a, 2.5, tensor_b)`. Only positions 0 and 2 are Tensors; position 1 holds a float scalar.

**Step 2 — Enumerate with original indices.** Using `enumerate((tensor_a, 2.5, tensor_b))` gives us `(0, tensor_a)`, `(1, 2.5)`, `(2, tensor_b)`.

**Step 3 — Filter to Tensors only.** The dict comprehension `{idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}` keeps only the Tensor entries. Position 1 (the float) is dropped entirely.

**Step 4 — Verify with print.** We print the dict to confirm the keys are `{0, 2}`, not `{0, 1}`. The 2.5 float never appears. This is critical because the backward dispatch uses `(forward_fn, argnum)` — wrong indices break it.

In [ ]:
import torch as t
from dataclasses import dataclass
from typing import Any

@dataclass
class MiniTensor:
    array: Any
    grad: Any = None

def build_parents_three_arg(args: tuple) -> dict:
    """Build parents dict; skip non-MiniTensor args, preserve original argidx."""
    return {idx: a for idx, a in enumerate(args) if isinstance(a, MiniTensor)}

# Three-arg scenario: tensor, scalar, tensor
t.manual_seed(3)
ta = MiniTensor(t.randn(3))
scalar = 2.5
tb = MiniTensor(t.randn(3))

args = (ta, scalar, tb)
parents = build_parents_three_arg(args)

print(f'Keys: {list(parents.keys())}')       # [0, 2] — 1 is absent
print(f'Key 0 is ta: {parents[0] is ta}')    # True
print(f'Key 2 is tb: {parents[2] is tb}')    # True
print(f'Float excluded: {1 not in parents}') # True